In [1]:
import os 
import pandas as pd
import numpy as np

from tsmd_evaluation.benchmark_generation import generate_random_walk
from sub_tsmd import load_validation, load_test

In [2]:
path = '../'
benchmark_path = f'{path}/noisy-attributes/'
seed = 42

In [3]:
nb_noisy_dimensions = [2**i for i in range(3, 7)]
nb_noisy_dimensions

[8, 16, 32, 64]

In [4]:
DATASETS = {
    'synthetic': [
        # Only use one variant
        's1'
    ]
} 

In [5]:
def add_random_walks(X, n):
    random_walks = np.array([generate_random_walk(X.shape[0]) for _ in range(n)])
    return np.concatenate([X, random_walks.T], axis=1)


def increase_ground_truth_mask(y, n):
    y_updated = []
    for motif_set_mask, motif_set_locations in y:
        updated_mask = np.zeros(shape=(motif_set_mask.shape[0] + n), dtype=bool)
        updated_mask[:motif_set_mask.shape[0]] = motif_set_mask
        y_updated.append((updated_mask, motif_set_locations))
    return y_updated


metadata = []
for origin in DATASETS:
    metadata_origin = pd.read_csv(f'{path}/{origin}/metadata.csv', index_col='ds_name')
    
    for ds_name in DATASETS[origin]:
        
        # Load the data       
        metadata_ds = metadata_origin.loc[ds_name]
        X_train, y_train = load_validation(f'{path}/{origin}/{ds_name}')
        X_test, y_test = load_test(f'{path}/{origin}/{ds_name}')

        for nb_dimensions in nb_noisy_dimensions:
            
            name = f'{ds_name}@{origin}#{nb_dimensions}-noisy'.lower()
            print(name)
                        
            # Generate the noisy train data
            np.random.seed(seed)
            X_train_noisy = [add_random_walks(X_train[i], nb_dimensions) for i in range(len(X_train))]
            
            # Generate the noisy test data
            np.random.seed(seed)
            X_test_noisy = [add_random_walks(X_test[i], nb_dimensions) for i in range(len(X_test))]
            
            # Update the masks of the ground truth
            y_train_noisy = [increase_ground_truth_mask(y_train[i], nb_dimensions) for i in range(len(y_train))]
            y_test_noisy = [increase_ground_truth_mask(y_test[i], nb_dimensions) for i in range(len(y_test))]
            
            # Save the noisy data
            path_to_benchmark_dataset = f'{benchmark_path}/{name}'
            if not os.path.exists(path_to_benchmark_dataset):
                os.mkdir(path_to_benchmark_dataset) 
            benchmark_train = pd.DataFrame(data={'ts': X_train_noisy, 'gt': y_train_noisy})
            benchmark_train.to_pickle(os.path.join(path_to_benchmark_dataset, 'validation.pkl'))
            benchmark_test = pd.DataFrame(data={'ts': X_test_noisy, 'gt': y_test_noisy})
            benchmark_test.to_pickle(os.path.join(path_to_benchmark_dataset, 'test.pkl'))
            
            # Set the meta data  
            metadata_ds = metadata_origin.loc[ds_name].copy()
            metadata_ds.at['D'] = X_train_noisy[0].shape[1]
            # Add property 'D_noisy', just after the 'D' property for easy interpretation
            keys = list(metadata_ds.index)
            values = list(metadata_ds.values)
            keys.insert(keys.index('D') + 1, 'D_noise')
            values.insert(keys.index('D') + 1, nb_dimensions)
            keys.insert(0, 'base_ds_name')  # To more easily find the hyperparameters
            values.insert(0, ds_name)
            metadata_ds = pd.Series(values, index=keys, name=name)
            metadata.append(metadata_ds)
            
metadata = pd.concat(metadata, axis=1).T
metadata.index.name = 'ds_name'

base@synthetic#8-noisy
base@synthetic#16-noisy
base@synthetic#32-noisy
base@synthetic#64-noisy


In [6]:
metadata = metadata.astype({'l_min': int, 'l_max': int, 'D': int, 'g_max': int, 'r': float, 'n_avg_train': float, 'n_avg_test': float, 'n_avg': float})
metadata.to_csv(os.path.join(benchmark_path, 'metadata.csv'))
metadata

,base_ds_name,l_min,l_max,D,D_noise,g_max,r,n_avg_train,n_avg_test,n_avg
ds_name,,,,,,,,,,
base@synthetic#8-noisy,base,50,50,14,8,5,3.6272,1190.34,1154.435,1161.616
base@synthetic#16-noisy,base,50,50,22,16,5,3.6272,1190.34,1154.435,1161.616
base@synthetic#32-noisy,base,50,50,38,32,5,3.6272,1190.34,1154.435,1161.616
base@synthetic#64-noisy,base,50,50,70,64,5,3.6272,1190.34,1154.435,1161.616
